# Notebook 05 (Exp 2) — Full Fine-Tuning (FFT)

Fixes vs Exp 1:
- **`learning_rate=5e-5`** — exp1 used 2e-5; both train and val loss plateaued around 1.05 indicating underfitting
- Raising LR 2.5x should allow the model to actually descend faster
- All other hyperparams identical to exp1

**Model**: Qwen2.5-1.5B-Instruct (same as exp1; 1.5B keeps FFT within 8GB VRAM)  
**Output**: `outputs/exp2/fft/`

In [1]:
import sys
import gc
import json
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('..').resolve()))
from src.model_utils import load_model_and_tokenizer, print_model_info
from src.data_utils import load_jsonl_as_hf_dataset
from src.training_utils import build_fft_training_args_exp2

from trl import SFTTrainer

ROOT          = Path('..').resolve()
PROCESSED_DIR = ROOT / 'data' / 'processed'
HF_CACHE      = ROOT / 'data' / 'raw' / 'huggingface_cache'
OUTPUT_DIR    = ROOT / 'outputs' / 'exp2' / 'fft'
FIG_DIR       = ROOT / 'outputs' / 'exp2' / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

print('Setup complete.')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


Setup complete.
GPU:  NVIDIA RTX PRO 5000 Blackwell
VRAM: 50.8 GB


## 1. Load Datasets

In [2]:
train_dataset = load_jsonl_as_hf_dataset(PROCESSED_DIR / 'train.jsonl')
val_dataset   = load_jsonl_as_hf_dataset(PROCESSED_DIR / 'val.jsonl')
print('Train:', train_dataset)
print('Val:  ', val_dataset)


Train: Dataset({
    features: ['messages'],
    num_rows: 40084
})
Val:   Dataset({
    features: ['messages'],
    num_rows: 2234
})


## 2. Load Model (BF16, All Parameters Trainable)

In [ ]:
model, tokenizer = load_model_and_tokenizer(
    MODEL_ID,
    quantization=None,
    attn_implementation='eager',
)

print_model_info(model)

if torch.cuda.is_available():
    print(f'VRAM after model load: {torch.cuda.memory_allocated(0)/1e9:.2f} GB')


## 3. Training Configuration (Exp 2 — LR=5e-5)

In [ ]:
training_args = build_fft_training_args_exp2(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    max_length=256,
    optim='paged_adamw_32bit',
)

print('Key FFT training arguments (Exp 2):')
print(f'  per_device_train_batch_size:  {training_args.per_device_train_batch_size}')
print(f'  gradient_accumulation_steps:  {training_args.gradient_accumulation_steps}')
print(f'  effective batch size:         {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'  learning_rate:                {training_args.learning_rate}  (exp1: 2e-5, exp2: 5e-5)')
print(f'  optimizer:                    {training_args.optim}')


## 4. Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

# Check if complete, or find latest checkpoint to resume
_final = OUTPUT_DIR / 'final_model'
if _final.exists():
    print(f'Training already complete — found {_final.name}')
    print('Delete final_model/ to re-train.')
    train_result = None
else:
    _ckpts = sorted(
        (OUTPUT_DIR / 'checkpoints').glob('checkpoint-*'),
        key=lambda p: int(p.name.split('-')[-1])
    ) if (OUTPUT_DIR / 'checkpoints').exists() else []
    _resume = str(_ckpts[-1]) if _ckpts else None
    print(f'Resuming from: {_resume}' if _resume else 'Starting FFT training (Exp 2, LR=5e-5) from scratch ...')
    train_result = trainer.train(resume_from_checkpoint=_resume)
    print(f'\nTraining complete.')
    print(f'  Train loss: {train_result.training_loss:.4f}')
    print(f'  Steps:      {train_result.global_step}')

## 5. Save Full Fine-Tuned Model

In [ ]:
model_path = str(OUTPUT_DIR / 'final_model')
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)
print(f'FFT model (Exp 2) saved to: {model_path}')


In [ ]:
log_path = OUTPUT_DIR / 'training_logs' / 'fft_log_history.json'
log_path.parent.mkdir(parents=True, exist_ok=True)
if train_result is not None:
    with open(log_path, 'w') as f:
        json.dump(trainer.state.log_history, f)
    print(f'Log history saved to: {log_path}')
else:
    print('Training was skipped — log not updated.')
    if log_path.exists(): print(f'Existing log at: {log_path}')

## 6. Training Loss Curve

In [ ]:
if train_src is not None:
    label = 'Train Loss' if has_train else 'Train Loss (TensorBoard)'
    ax.plot(train_src['step'], train_src['loss'], label=label, color='steelblue')

if eval_tb is not None and not eval_tb.empty:
    ax.plot(eval_tb['step'], eval_tb['eval_loss'],
            label='Val Loss', color='darkorange', linestyle='--')
    best = eval_tb.loc[eval_tb['eval_loss'].idxmin()]
    ax.axvline(best['step'], color='green', linestyle=':', alpha=0.7,
               label=f'Best val {best["eval_loss"]:.4f} @ step {int(best["step"])}')

ax.set_xlabel('Step'); ax.set_ylabel('Loss')
ax.set_title('QLoRA Training — Loss Curve (Exp 2, 2 epochs + EarlyStopping)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'exp2_lora_loss_curve.png', dpi=150)
plt.show()
print(f'Saved → {FIG_DIR / "exp2_lora_loss_curve.png"}')

In [ ]:
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print('Memory freed.')
